# Analyse: Personenanzahl


In [1]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from retinaface import RetinaFace

DATA_DIR = Path('../../data')
COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '12_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_structural_personen_anzahl.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'
SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 24
MAX_FRAME_WIDTH = 640

df = pd.read_csv(COMBINED_CSV)
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()


print(f'Loaded {len(df)} rows from {COMBINED_CSV.name}')



Loaded 500 rows from 01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv


In [2]:
METRIC_COLUMNS = [
    'personen_anzahl',
    'personen_anzahl_max',
    'detected_person_frames',
    'sampled_frames_personen_anzahl',
]

def get_duration_seconds(row):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan

def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()

df['has_frames'] = [has_frames(video_id) for video_id in df[video_id_col].astype(str)]
missing_ids = df.loc[~df['has_frames'], video_id_col].astype(str).tolist()
print(f"Videos with frame folder: {df['has_frames'].sum()}")
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')

hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())


Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


In [3]:
def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.png'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, default_max_frames, len(frame_files)))
    indices = np.linspace(0, len(frame_files) - 1, num=target_frames, dtype=int)
    return [frame_files[idx] for idx in np.unique(indices)]

def prepare_frame(image_path: Path):
    image = cv2.imread(str(image_path))
    if image is None:
        return None
    height, width = image.shape[:2]
    if width > MAX_FRAME_WIDTH:
        scale = MAX_FRAME_WIDTH / width
        image = cv2.resize(image, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
    return image

def count_faces(image_path: Path) -> int:
    try:
        result = RetinaFace.detect_faces(str(image_path))
        if isinstance(result, dict):
            return len(result)
        return 0
    except Exception:
        return 0

def count_people_hog(image: np.ndarray) -> int:
    if image is None:
        return 0
    rects, _ = hog.detectMultiScale(image, winStride=(8, 8), padding=(8, 8), scale=1.05)
    return int(len(rects)) if rects is not None else 0

def estimate_people_in_frame(image_path: Path) -> int:
    image = prepare_frame(image_path)
    body_count = count_people_hog(image)
    face_count = count_faces(image_path)
    return int(max(body_count, face_count))


In [4]:
personen_anzahl_mean = []
personen_anzahl_max = []
detected_person_frames = []
sampled_frames_count = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Processing videos'):
    video_id = str(row[video_id_col])
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(video_id, duration_seconds=duration_seconds)

    counts = [estimate_people_in_frame(fp) for fp in frame_paths]
    valid_counts = [c for c in counts if pd.notna(c)]
    detected_frames = sum(1 for c in valid_counts if c > 0)

    if valid_counts:
        personen_anzahl_mean.append(float(np.mean(valid_counts)))
        personen_anzahl_max.append(float(np.max(valid_counts)))
    else:
        personen_anzahl_mean.append(np.nan)
        personen_anzahl_max.append(np.nan)
  
    detected_person_frames.append(int(detected_frames))
    sampled_frames_count.append(int(len(frame_paths)))


Processing videos: 100%|██████████| 500/500 [15:41:08<00:00, 112.94s/it]  


In [5]:
df['personen_anzahl'] = personen_anzahl_mean
df['personen_anzahl_max'] = personen_anzahl_max
df['detected_person_frames'] = detected_person_frames
df['sampled_frames_personen_anzahl'] = sampled_frames_count

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

print(df[['influencer_type', 'personen_anzahl', 'personen_anzahl_max', 'detected_person_frames']].groupby('influencer_type').agg(['count', 'mean', 'std']).round(4))
print(f'Wrote {OUTPUT_CSV} with shape {df.shape}')
df[['video_id', 'influencer_type', 'personen_anzahl', 'personen_anzahl_max', 'detected_person_frames', 'sampled_frames_personen_anzahl']].head(20)


                personen_anzahl                 personen_anzahl_max         \
                          count    mean     std               count   mean   
influencer_type                                                              
ai                          250  2.3325  1.5947                 250  4.440   
real                        250  1.9311  0.8690                 250  4.668   

                        detected_person_frames                  
                    std                  count    mean     std  
influencer_type                                                 
ai               2.6018                    250  13.476  6.0268  
real             4.8152                    250  17.144  6.6938  
Wrote ..\..\data\04_analysis_results\visual_features\12_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_structural_personen_anzahl.csv with shape (500, 49)


,video_id,influencer_type,personen_anzahl,personen_anzahl_max,detected_person_frames,sampled_frames_personen_anzahl
0,7516243181650988334,ai,2.142857,3.0,7,7
1,7515925552549678378,ai,2.500000,5.0,10,10
2,7521159314757684535,ai,2.375000,6.0,8,8
3,7521486299098778894,ai,5.875000,11.0,7,8
4,7521490969468865847,ai,2.625000,5.0,8,8
5,7516637350072470798,ai,1.250000,2.0,8,8
6,7516241082754010414,ai,1.000000,1.0,8,8
7,7516244049251208494,ai,1.125000,2.0,8,8
8,7516955006088613175,ai,1.000000,1.0,6,6
9,7516376664570416426,ai,1.000000,1.0,8,8
